# 04 — Perbandingan Model: v1 (ConvNeXt 11 kelas) vs v2 (ConvNeXt 22 kelas) vs v3 (CLIP frozen)

Satu tempat untuk slide "perbandingan model" presentasi: baca
`metrics_report.json` tiap run langsung dari disk (bukan angka di-hardcode
di notebook ini), tampilkan tabel + chart perbandingan, dan tegaskan
batasan perbandingan (jumlah kelas beda, status training beda) supaya
tidak overclaim di presentasi.

**Prasyarat**: setidaknya salah satu dari `outputs/v1_11class_backup/metrics_report.json`,
`outputs/v2_22class/metrics_report.json`, `outputs/v3_clip/metrics_report.json`
sudah ada (jalankan `01_train.ipynb` di folder model yang relevan dulu kalau
belum). Sel di bawah TIDAK error kalau salah satu belum ada -- baris itu
ditampilkan sbg "belum dijalankan", bukan menggagalkan seluruh notebook.

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
import json

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml-visual-search/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import resolve_path

print("working dir:", os.getcwd())

## 2. Baca metrics_report.json tiap run

Tabel ini DIISI OTOMATIS dari file di disk -- kalau salah satu run belum
dijalankan/dievaluasi, kolom itu tampil `None` (bukan menebak angka).

In [ ]:
RUNS = [
    {"id": "v1_11class", "label": "v1 -- ConvNeXt fine-tuned (11 kelas, 5.659 gambar)",
     "metrics_path": "outputs/v1_11class_backup/metrics_report.json",
     "dataset": "5.659 gambar", "n_classes": 11, "arch": "convnext_small (fine-tune_partial)",
     "tuning": "Optuna 15-trial"},
    {"id": "v2_22class", "label": "v2 -- ConvNeXt fine-tuned (22 kelas, 33.937 gambar)",
     "metrics_path": "outputs/v2_22class/metrics_report.json",
     "dataset": "33.937 gambar", "n_classes": 22, "arch": "convnext_small (fine-tune_partial)",
     "tuning": "reuse hasil tuning v1"},
    {"id": "v3_clip", "label": "v3 -- CLIP frozen + linear probe (22 kelas, 33.937 gambar)",
     "metrics_path": "outputs/v3_clip/metrics_report.json",
     "dataset": "33.937 gambar", "n_classes": 22, "arch": "CLIP ViT-B-32 (frozen) + Linear head",
     "tuning": "n/a (reuse hyperparameter classifier head)"},
]

rows = []
for r in RUNS:
    p = resolve_path(r["metrics_path"])
    m = json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
    rows.append({
        "model": r["id"],
        "arsitektur": r["arch"],
        "dataset": r["dataset"],
        "n_kelas": r["n_classes"],
        "tuning": r["tuning"],
        "test_accuracy": m["accuracy"] if m else None,
        "test_macro_f1": m["macro_f1"] if m else None,
        "test_weighted_f1": m["weighted_f1"] if m else None,
        "status": "selesai & dievaluasi" if m else "BELUM DIJALANKAN -- jalankan 01_train.ipynb dulu",
    })

df = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_colwidth", 60)
df

## 3. Chart perbandingan test macro-F1

**PERINGATAN penting (lihat juga `notebooks/README.md`)**: macro-F1 v1 (11
kelas) TIDAK apple-to-apple dgn v2/v3 (22 kelas) -- baseline tebak-acak utk
11 kelas (~9%) jauh lebih tinggi dari 22 kelas (~4,5%), jadi skor absolut
LEBIH TINGGI di 11 kelas sebagian karena soal lebih mudah, bukan murni model
lebih baik. Chart ini TETAP ditampilkan utk transparansi bahan presentasi,
tapi JANGAN disajikan sbg "makin tinggi makin bagus" tanpa catatan ini.

In [ ]:
plot_df = df.dropna(subset=["test_macro_f1"])
if len(plot_df) == 0:
    print("Belum ada run yang dievaluasi -- jalankan 01_train.ipynb di salah satu folder model dulu.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(plot_df.index, plot_df["test_macro_f1"], color=["tab:blue", "tab:orange", "tab:green"][:len(plot_df)])
    for b, v in zip(bars, plot_df["test_macro_f1"]):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=10)
    ax.set_ylim(0, 1); ax.set_ylabel("test macro-F1")
    ax.set_title("Perbandingan test macro-F1 -- HATI-HATI beda jumlah kelas (lihat catatan di atas)")
    plt.tight_layout(); plt.show()

## 4. Tabel hyperparameter per model (untuk slide)

In [ ]:
hyperparam_rows = [
    {"model": "v1_11class", "learning_rate": 0.0002584949196006611, "backbone_lr_mult": 0.18427033274320545,
     "weight_decay": 0.0009849813063176139, "dropout": 0.2017262123368978, "unfreeze_frac": 0.4406353983164585,
     "sumber": "Optuna TPE, trial #13/15 (val macro-F1 0.7592)"},
    {"model": "v2_22class", "learning_rate": 0.0002584949196006611, "backbone_lr_mult": 0.18427033274320545,
     "weight_decay": 0.0009849813063176139, "dropout": 0.2017262123368978, "unfreeze_frac": 0.4406353983164585,
     "sumber": "reuse hasil tuning v1 (tidak di-tuning ulang)"},
]
hp_df = pd.DataFrame(hyperparam_rows).set_index("model")
print(hp_df.to_string())

v3_report = resolve_path("outputs/v3_clip/metrics_report.json")
if v3_report.exists():
    m3 = json.loads(v3_report.read_text(encoding="utf-8"))
    print("\nv3_clip classifier head config:")
    print(json.dumps(m3.get("classifier_config", {}), indent=2))
else:
    print("\nv3_clip: belum dijalankan -- jalankan notebooks/v3_clip/01_train.ipynb dulu.")

## 5. Model terbaik -- rekomendasi untuk presentasi

Isi sel ini SETELAH ketiga run (terutama v2 & v3) selesai dievaluasi --
jangan isi manual sebelum datanya ada. Kerangka argumen yang sudah
disiapkan (lihat `notebooks/README.md` untuk detail kronologi):

- **Kalau tujuannya "model terbaik utk live di backend sekarang"**: bukan
  murni soal macro-F1 tertinggi -- v2 sudah terbukti 8/8 benar di 8 gambar
  katalog produksi riil meski formal belum selesai training/evaluasi test
  set, sedangkan v1 (macro-F1 test lebih tinggi, 0.78) cuma 5/8 di tes yang
  sama. Jelaskan ke audiens KENAPA ini bisa terjadi (22 kelas vs 11 kelas,
  baseline acak beda, data katalog produksi vs test set akademik).
- **Kalau tujuannya "model paling murah/efisien dilatih"**: v3 (CLIP frozen)
  -- bandingkan `test_macro_f1` section 2 vs biaya training (menit vs jam,
  lihat catatan di `v3_clip/01_train.ipynb` section 8).
- **Kalau tujuannya "upgrade paling menjanjikan ke depan"**: tergantung hasil
  v2 vs v3 di section 2/3 setelah KEDUANYA selesai dievaluasi -- isi
  kesimpulan di sini manual setelah lihat angkanya, jangan sebelum itu.